In [ ]:
board = input("Enter 9 cells (X, O, or .): ").split()
wins = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]

def winner(b):
    for x, y, z in wins:
        if b[x] != "." and b[x] == b[y] == b[z]:
            return b[x]
    return None

def alpha_beta(b, p, alpha, beta):
    if winner(b) == "X": return 1, None
    if winner(b) == "O": return -1, None
    if "." not in b: return 0, None
    best = (-2 if p == "X" else 2, None)
    for i in range(9):
        if b[i] == ".":
            b[i] = p
            score, _ = alpha_beta(b, "O" if p == "X" else "X", alpha, beta)
            b[i] = "."
            if p == "X":
                best = max(best, (score, i))
                alpha = max(alpha, best[0])
            else:
                best = min(best, (score, i))
                beta = min(beta, best[0])
            if alpha >= beta:
                break
    return best
score, move = alpha_beta(board, "X", -2, 2)
print("Board:", [board[:3], board[3:6], board[6:]])
if move is not None:
    print("Best move (row, column):", (move // 3 + 1, move % 3 + 1))
print("Score:", score)


In [ ]:
from heapq import heappop, heappush

start = tuple(map(int, input("Enter 9 values (0 for blank): ").split()))
goal = (1, 2, 3, 4, 5, 6, 7, 8, 0)
def distance(board):
    return sum(abs(i//3 - (v-1)//3) + abs(i%3 - (v-1)%3) for i, v in enumerate(board) if v)
def a_star(start):
    pq = [(distance(start), 0, start, [start])]
    vis = set()
    while pq:
        f, g, b, path = heappop(pq)
        if b == goal:
            return path
        if b in vis:
            continue
        vis.add(b)
        z = b.index(0)
        for d in (-3, 3, -1, 1): # Up, Down, Left, Right
            nz = z + d
            if 0 <= nz < 9 and not (z%3 == 2 and d == 1) and not (z%3 == 0 and d == -1):
                nb = list(b)
                nb[z], nb[nz] = nb[nz], nb[z]
                nb = tuple(nb)
                heappush(pq, (g + 1 + distance(nb), g + 1, nb, path + [nb]))
    return []
path = a_star(start)
print("Solution found:", bool(path))
if path:
    print("Number of moves:", len(path) - 1)
    for step, board in enumerate(path):
        print(f"Step {step}:", [board[:3], board[3:6], board[6:]])


In [ ]:
def reply(message):
    message = message.lower()
    for symbol in ".,!?":
        message = message.replace(symbol, " ")
    words = message.split()
    if "return" in words or "refund" in words:
        return "You can return unused items within 30 days."
    if "track" in words or "status" in words:
        return "Open My Orders and select Track Package."
    if "shipping" in words or "delivery" in words:
        return "Standard delivery takes 3 to 5 business days."
    if "hours" in words or "open" in words:
        return "Support is open from 9 AM to 6 PM, Monday to Friday."
    if "hello" in words or "hi" in words:
        return "Hello! I can help with orders, delivery, returns or support hours."
    if "bye" in words:
        return "Goodbye!"
    return "Please ask about order tracking, delivery, returns or support hours."
while True:
    msg = input("Customer: ")
    print("Bot:", reply(msg))
    if "bye" in msg.lower():
        break


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("iris.csv")
X = StandardScaler().fit_transform(df.iloc[:, :-1])
inertia = [KMeans(n_clusters=k, random_state=42).fit(X).inertia_ for k in range(1, 11)]
plt.plot(range(1, 11), inertia, marker="o")
plt.title("Elbow Method")
plt.xlabel("K")
plt.ylabel("Inertia")
plt.show()
clusters = KMeans(n_clusters=3, random_state=42).fit_predict(X)
print("Cluster sizes:\n", pd.Series(clusters).value_counts())
plt.scatter(df.iloc[:, 2], df.iloc[:, 3], c=clusters)
plt.xlabel("Petal Length")
plt.ylabel("Petal Width")
plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
df = pd.read_csv("iris.csv")
X = StandardScaler().fit_transform(df.drop(columns="species"))
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
pred = model.predict(X_test)
cm = confusion_matrix(y_test, pred)
acc = accuracy_score(y_test, pred)
print("Confusion Matrix:\n", cm)
print(f"Accuracy: {acc:.2f} | Error Rate: {1 - acc:.2f}\n")
for i, name in enumerate(model.classes_):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    tn = cm.sum() - (tp + fp + fn)
    print(f"{name} -> TP:{tp}, FP:{fp}, TN:{tn}, FN:{fn} | Precision:{tp/(tp+fp):.2f}, Recall:{tp/(tp+fn):.2f}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("BostonHousing.csv")
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pred = LinearRegression().fit(X_train, y_train).predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)
print("R2 Score:", r2_score(y_test, pred))
plt.scatter(y_test, pred)
plt.xlabel("Actual Price ($1000)")
plt.ylabel("Predicted Price ($1000)")
plt.title("Actual vs Predicted Home Prices")
plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
data = pd.read_csv("Social_Network_Ads.csv")
X = data[["Age", "EstimatedSalary"]]
y = data["Purchased"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
model = LogisticRegression()
model.fit(X_train, y_train)
predicted = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predicted))
print("Precision:", precision_score(y_test, predicted))
print("Recall:", recall_score(y_test, predicted))
print("Confusion matrix:\n", confusion_matrix(y_test, predicted))


In [ ]:
board = input("Enter 9 cells (X, O, or .): ").split()
wins = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]
def winner(b):
    for x, y, z in wins:
        if b[x] != "." and b[x] == b[y] == b[z]:
            return b[x]
    return None
def minimax(b, p):
    if winner(b) == "X": return 1, None
    if winner(b) == "O": return -1, None
    if "." not in b: return 0, None
    moves = []
    for i in range(9):
        if b[i] == ".":
            b[i] = p
            score, _ = minimax(b, "O" if p == "X" else "X")
            b[i] = "."
            moves.append((score, i))
    return max(moves) if p == "X" else min(moves)
score, move = minimax(board, "X")
print("Board:", [board[:3], board[3:6], board[6:]])
if move is not None:
    print("Best move (row, column):", (move // 3 + 1, move % 3 + 1))
print("Score:", score)


In [ ]:
n = int(input("Enter N: "))
sols = []

def solve(b, cols, d1, d2):
    r = len(b)
    if r == n:
        sols.append(b)
        return
    for c in range(n):
        if c not in cols and r - c not in d1 and r + c not in d2:
            solve(b + [c], cols | {c}, d1 | {r - c}, d2 | {r + c})

solve([], set(), set(), set())
print("Number of solutions:", len(sols))

if sols:
    print("Queen columns:", [c + 1 for c in sols[0]])
    for c in sols[0]:
        print(" ".join("Q" if i == c else "." for i in range(n)))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv("wine.csv")
X = StandardScaler().fit_transform(df.iloc[:, 1:])
y = df.iloc[:, 0]

pca = PCA(n_components=2)
pc = pca.fit_transform(X)

print("Original shape:", X.shape, "| Reduced shape:", pc.shape)
print("Explained variance:", pca.explained_variance_ratio_)

plt.scatter(pc[:, 0], pc[:, 1], c=y)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA on Wine Dataset")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("titanic.csv")
print("Shape:", df.shape, "\nMissing:\n", df.isna().sum())
print(df.describe(include="all"))
df["age"] = df["age"].fillna(df["age"].median())
df["deck"] = df["deck"].fillna("Unknown")
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop_duplicates()
for col in ["age", "fare"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
df["fare_log"] = np.log1p(df["fare"])
plt.hist(df["fare_log"])
plt.title("Log Transformed Fare")
plt.show()

df = df.drop(columns=["alive", "class", "embark_town", "who", "adult_male", "alone"], errors="ignore")
encoded = pd.get_dummies(df, columns=["pclass", "sex", "embarked", "deck"], dtype=int)
print("Encoded Shape:", encoded.shape)
print(encoded.head(3))


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("emails.csv").drop(columns="Email No.").drop_duplicates()
X = StandardScaler().fit_transform(df.drop(columns="Prediction"))
y = df["Prediction"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pred = SVC(kernel="linear").fit(X_train, y_train).predict(X_test)
cm = confusion_matrix(y_test, pred)
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix:\n", cm)
print(f"False Positives (Normal flagged as Spam): {fp} | False Negatives (Spam missed): {fn}\n")
print(classification_report(y_test, pred, target_names=["Not Spam", "Spam"]))
